In [1]:
%pip install pymupdf

   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/19.2 MB 3.8 MB/s eta 0:00:05
   --- ------------------------------------ 1.6/19.2 MB 4.9 MB/s eta 0:00:04
   ---- ----------------------------------- 2.4/19.2 MB 4.3 MB/s eta 0:00:04
   ------- -------------------------------- 3.7/19.2 MB 4.7 MB/s eta 0:00:04
   --------- ------------------------------ 4.5/19.2 MB 4.5 MB/s eta 0:00:04
   ---------- ----------------------------- 5.2/19.2 MB 4.4 MB/s eta 0:00:04
   ----------- ---------------------------- 5.5/19.2 MB 4.1 MB/s eta 0:00:04
   ----------- ---------------------------- 5.8/19.2 MB 3.7 MB/s eta 0:00:04
   ------------- -------------------------- 6.3/19.2 MB 3.6 MB/s eta 0:00:04
   -------------- ------------------------- 7.1/19.2 MB 3.5 MB/s eta 0:00:04
   --------------- ------------------------ 7.6/19.2 MB 3.5 MB/s eta 0:00:04
   ---------------- ----------------------- 8.1/19.2 MB 3.4 MB/s eta 0:00:04
   ---

In [2]:
!pip install pdfplumber

  Using cached pillow-12.2.0-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
  Using cached charset_normalizer-3.4.7-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached cryptography-48.0.0-cp311-abi3-win_amd64.whl.metadata (4.3 kB)
  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ------ --------------------------------- 1.0/6.6 MB 4.9 MB/s eta 0:00:02
   ----------- ---------------------------- 1.8/6.6 MB 4.3 MB/s eta 0:00:02
   -------------- ------------------------- 2.4/6.6 MB 3.9 MB/s eta 0:00:02
   ----------------- ---------------------- 2.9/6.6 MB 3.4 MB/s eta 0:00:02
   -------------------- ------------------- 3.4/6.6 MB 3.2 MB/s eta 0:00:01
   ------------------------- -------------- 4.2/6.6 MB 3.4 MB/s eta 0:00:01
   ------------------------------- -------- 5.2/6.6 MB 3.6 MB/s eta 0:00:01
   ----------------------------

## Extracting text from PDF

In [3]:
import fitz
import pdfplumber

FOOTER_MARGIN = 50
HEADER_MARGIN = 70

def extract_text(path):
    doc = fitz.open(path)

    header_content = []
    footer_content = []
    main_content = ""
    all_tables_json = []
    image_count = 0

    with pdfplumber.open(path) as pdf:
        for page_num in range(len(doc)):
            page = doc[page_num]
            plumber_page = pdf.pages[page_num]

            page_height = page.rect.height
            footer_y_start = page_height - FOOTER_MARGIN
            header_y_end = HEADER_MARGIN

            blocks = page.get_text("blocks")
            blocks = sorted(blocks, key=lambda b: (b[1], b[0]))

            tables = plumber_page.find_tables()
            table_regions = []

            # Extract tables
            for table in tables:
                table_regions.append(table.bbox)

                extracted = table.extract()

                if extracted and len(extracted) > 1:
                    headers = [
                        h.replace("\n", " ").strip() if h else f"column_{i}"
                        for i, h in enumerate(extracted[0])
                    ]

                    table_json = []

                    for row in extracted[1:]:
                        cleaned_row = [
                            cell.replace("\n", " ").strip() if cell else ""
                            for cell in row
                        ]

                        row_dict = dict(zip(headers, cleaned_row))
                        table_json.append(row_dict)

                    all_tables_json.append({
                        "page": page_num + 1,
                        "table_data": table_json
                    })

            # Extract text blocks
            for block in blocks:
                x0, y0, x1, y1, text = block[:5]
                text = text.strip()

                if not text:
                    continue

                # HEADER JSON
                if y1 <= header_y_end:
                    header_content.append({
                        "page": page_num + 1,
                        "text": text
                    })
                    continue

                # FOOTER JSON
                if y0 >= footer_y_start:
                    footer_content.append({
                        "page": page_num + 1,
                        "text": text
                    })
                    continue

                # Skip table text from PyMuPDF
                inside_table = False
                for tx0, ty0, tx1, ty1 in table_regions:
                    if y0 >= ty0 and y1 <= ty1:
                        inside_table = True
                        break

                if inside_table:
                    continue

                main_content += text + "\n\n"

            image_count += len(page.get_images())

    return header_content, main_content, footer_content, all_tables_json, image_count


# Run
path = "data/finance_evaluation.pdf"

header_content, main_content, footer_content, tables_json, image_count = extract_text(path)

In [4]:
header_content

[{'page': 1, 'text': 'PAGE 1 OF 3'},
 {'page': 3, 'text': 'PAGE 2 OF 3'},
 {'page': 4, 'text': 'PAGE 3 OF 3'}]

In [5]:
footer_content

[{'page': 1,
  'text': 'Apex Global Wealth Management • Q1 2026 Portfolio Review\nPage 1 of 4'},
 {'page': 2,
  'text': 'Apex Global Wealth Management • Q1 2026 Portfolio Review\nPage 2 of 4'},
 {'page': 3,
  'text': 'Apex Global Wealth Management • Q1 2026 Portfolio Review\nPage 3 of 4'},
 {'page': 4,
  'text': 'Apex Global Wealth Management • Q1 2026 Portfolio Review\nPage 4 of 4'}]

In [6]:
main_content

'DOCUMENT HEADER: QUARTERLY FINANCIAL REVIEW\n\nISSUER: APEX GLOBAL WEALTH MANAGEMENT GROUP\n\nREPORTING PERIOD: Q1 2026\n\nTARGET AUDIENCE: PRIVATE WEALTH PORTFOLIO CLIENTS\n\nSECTION 1: MONETARY POLICY AND CAPITAL MARKETS CONTEXT\n\nThe first quarter of 2026 has demonstrated remarkable resilience across global capital markets, characterized by a\n\ntransition toward normalizing monetary policies and steady corporate earnings growth. Central banks globally have\n\ninitiated a measured approach to interest rate adjustments, easing structural pressures on both fixed income and\n\nequity valuations. While macroeconomic uncertainties linger regarding sticky supply-chain components, consumer\n\nspending metrics remain healthy, supporting a baseline expansion model. In this dynamic environment, our core\n\nasset allocation strategy prioritized high-quality, dividend-yielding equities paired with opportunistic fixed-income\n\ndurations. This dual-pronged focus allowed our managed portfolios 

In [8]:
tables_json

[{'page': 2,
  'table_data': [{'Asset Classification': 'Domestic Large-Cap Equities',
    'Target Alloc %': '30.0%',
    'Current Alloc %': '32.4%',
    'Current Value ($)': '324,000.00',
    'Q1 Return (%)': '+5.8%',
    'Strategic Action': 'Trim to Target'},
   {'Asset Classification': 'International Developed Equities',
    'Target Alloc %': '15.0%',
    'Current Alloc %': '14.2%',
    'Current Value ($)': '142,000.00',
    'Q1 Return (%)': '+1.2%',
    'Strategic Action': 'Maintain Position'},
   {'Asset Classification': 'Emerging Markets Equities',
    'Target Alloc %': '5.0%',
    'Current Alloc %': '4.1%',
    'Current Value ($)': '41,000.00',
    'Q1 Return (%)': '-2.4%',
    'Strategic Action': 'Opportunistic Add'},
   {'Asset Classification': 'Investment-Grade Corporate Bonds',
    'Target Alloc %': '25.0%',
    'Current Alloc %': '24.5%',
    'Current Value ($)': '245,000.00',
    'Q1 Return (%)': '+0.9%',
    'Strategic Action': 'Reinvest Coupons'},
   {'Asset Classificatio

In [ ]:
!pip install faiss-cpu ollama

  Using cached faiss_cpu-1.13.2-cp313-cp313-win_amd64.whl.metadata (7.6 kB)
Using cached faiss_cpu-1.13.2-cp313-cp313-win_amd64.whl (18.9 MB)
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
    --------------------------------------- 0.3/12.3 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.3 MB 1.7 MB/s eta 0:00:07
   ---- ----------------------------------- 1.3/12.3 MB 2.0 MB/s eta 0:00:06
   ----- ---------------------------------- 1.8/12.3 MB 2.1 MB/s eta 0:00:05
   ------- -------------------------------- 2.4/12.3 MB 2.1 MB/s eta 0:00:05
   -------- ------------------------------- 2.6/12.3 MB 2.1 MB/s eta 0:00:05
   ----------- ---------------------------- 3.4/12.3 MB 2.2 MB/s eta 0:00:05
   ----------- ---------------------------- 3.7/12.3 MB 2.2 MB/s eta 0:00:04
   ------------- -------------------------- 4.2/12.3 MB 2.2 MB/s eta 0:00:04
   --------------- -------------

## RAG with ChromaDB

In [ ]:
!pip install chromadb sentence-transformers

In [ ]:
import json
import chromadb
from sentence_transformers import SentenceTransformer

# --- 1. Prepare documents from extracted content ---

documents = []
metadatas = []
ids = []

# Add main content chunks (split by double newline)
chunks = [c.strip() for c in main_content.split("\n\n") if c.strip()]
for i, chunk in enumerate(chunks):
    documents.append(chunk)
    metadatas.append({"source": "main_content", "chunk_index": i})
    ids.append(f"main_{i}")

# Add table data as stringified JSON chunks
for t_idx, table in enumerate(tables_json):
    for r_idx, row in enumerate(table["table_data"]):
        row_text = ", ".join(f"{k}: {v}" for k, v in row.items())
        documents.append(row_text)
        metadatas.append({"source": "table", "page": table["page"], "row": r_idx})
        ids.append(f"table_{t_idx}_row_{r_idx}")

print(f"Total documents to index: {len(documents)}")

In [ ]:
# --- 2. Embed with sentence-transformers ---

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(documents, show_progress_bar=True).tolist()

In [ ]:
# --- 3. Store in ChromaDB (in-memory) ---

chroma_client = chromadb.Client()  # in-memory; use chromadb.PersistentClient(path="./chroma_db") to persist

collection = chroma_client.get_or_create_collection(
    name="finance_rag",
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print(f"Indexed {collection.count()} documents into ChromaDB")

In [ ]:
# --- 4. RAG Query function ---

def rag_query(question: str, top_k: int = 3) -> str:
    """Retrieve top_k relevant chunks and return them as context."""
    query_embedding = embed_model.encode([question]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    retrieved_docs = results["documents"][0]
    context = "\n\n---\n\n".join(retrieved_docs)

    prompt = f"""You are a financial analyst assistant. Use the context below to answer the question.

Context:
{context}

Question: {question}
Answer:"""
    return prompt


# --- 5. Test retrieval ---
question = "What is the current allocation for Domestic Large-Cap Equities?"
prompt = rag_query(question)
print(prompt)

In [ ]:
# --- 6. (Optional) Generate answer with Ollama ---
# Requires Ollama running locally: https://ollama.com
# Pull a model first: ollama pull llama3.2

import ollama

def rag_answer(question: str, model: str = "llama3.2", top_k: int = 3) -> str:
    prompt = rag_query(question, top_k=top_k)
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]


answer = rag_answer("What is the Q1 return for Emerging Markets Equities?")
print(answer)